# Aurora local project dashboard

An executable project document for the FastAPI insulation enquiry agent. Start here in either checkout. **Run All is read-only by default:** no downloads, hosted AI, model pulls, database writes, rebuilds, tests or server startup.

Optional actions below are disabled unless you enter the exact approval text for that action. Reset approvals afterwards. Tests can create temporary/local database files; rebuilds update generated outputs. Nothing here synchronises checkouts or deletes, restores or overwrites downloaded TDS files.

This notebook controls project commands, not the coding assistant's billing. To avoid cloud coding credits, select a genuinely local provider in your editor/agent; local shell commands alone do not make a hosted assistant free.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import os
import subprocess
import sys

def find_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'web_agent.py').is_file() and (candidate / 'knowledge').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook from the repository root or its notebooks directory.')

ROOT = find_root()
print('Checkout:', ROOT)
print('Kernel Python:', sys.executable)
print('Use a kernel with this project\'s existing dependencies. No packages are installed here.')


## Git baseline and working changes

These commands use local Git metadata only: they do not fetch, push, switch branches or modify files. Cached remote refs may be stale. Keep `Aurora-V1` as a historical release marker, not a moving tag. Commit improvements in one checkout and integrate them deliberately; never overwrite a dirty checkout with a folder copy.


In [ ]:
def read_git(*args):
    result = subprocess.run(['git', '--no-pager', *args], cwd=ROOT, text=True, capture_output=True, check=False)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or f'Git exited {result.returncode}')
    print(result.stdout.strip())

read_git('status', '--short', '--branch')
read_git('log', '-5', '--oneline', '--decorate')


## Preserved TDS sources

Git does not carry downloaded PDFs. A new clone can have an empty TDS folder even when its code is current. The desktop source library is still needed: do not delete, move, rename or overwrite it. The paths below can be adjusted for another machine; scanning them is read-only and does not imply the copies are identical. Counts include shared documents, not one TDS per family.


In [ ]:
TDS_LIBRARY = Path(os.environ.get('AURORA_TDS_LIBRARY', str(Path.home() / 'Desktop' / 'Data Gathering' / 'data' / 'TDS')))
TDS_PATHS = [ROOT / 'data' / 'tds', ROOT / 'data' / 'tds_inbox', TDS_LIBRARY, Path.home() / 'Desktop' / 'Aurora-POC' / 'data' / 'tds']
for folder in TDS_PATHS:
    print('\n', folder)
    if not folder.is_dir():
        print('  Missing in this checkout/machine; do not fetch automatically.')
        continue
    documents = [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.pdf', '.docx'}]
    print('  Documents:', len(documents), '| MiB:', round(sum(p.stat().st_size for p in documents) / (1024 ** 2), 1))
    if documents:
        newest = max(documents, key=lambda p: p.stat().st_mtime)
        print('  Most recently modified:', newest)

manifest_path = ROOT / 'knowledge' / '_tds_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    print('\nTracked provenance entries:', len(manifest))
else:
    print('Tracked TDS manifest is missing.')


## Research progress and local data health

Read existing JSON and files only. Research status does not certify product identity, performance or compliance. See `BOT_POLICY.md` and `AUDIT_SECURITY.md`. Customer lead contents and secrets are deliberately not displayed here.


In [ ]:
statuses = Counter()
for record_path in sorted((ROOT / 'knowledge').glob('*/research/*.json')):
    record = json.loads(record_path.read_text(encoding='utf-8-sig'))
    statuses[record.get('status', 'status_missing')] += 1
print('Research records:', sum(statuses.values()))
for status, count in sorted(statuses.items()):
    print(f'  {status}: {count}')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import data_health
for result in data_health.run_all('authoring'):
    print(f'[{result.status.upper()}] {result.name}: {result.detail}')


## Optional actions: explicitly approved, offline commands

Choose **one** action and set `APPROVE_ACTION` to that action's exact name before running its cell. Empty means no action. Only the fixed commands listed below can run. No cloud scripts, downloads, TDS restore/copy operations or package installation are offered.

- `targeted-tests`: lead, conversation-service and FastAPI tests.
- `research-tests`: synthetic family-first research, named accounts, citations, reviews and preview/publish tests; no real evidence is promoted.
- `full-tests`: all tests (some integration cases may be skipped by their existing gates).
- `evaluate-conversations`: synthetic customer scenarios, model-off; writes a local transcript report using a temporary lead database.
- `knowledge-overview`, `knowledge-validation`, `function-map`: shared offline readers and static callable inventory.
- `source-audit-preview`: all-page extraction preview; no files written, fetching or approval.
- `validate-catalogue`, `validate-generated`: current authoring structure/projection checks. `validate-aircall`: optional legacy artifacts, not reviewed-release readiness.
- `retrieval-preview`, `missing-source-preview`: read-only drafts. Explicit CLI confirmation is required for intentional writes after authoring backup; manual fields/history are preserved.
- `legacy-rebuild-catalogue`: optional raw SQLite writer, not the primary workflow. Back up authoring inputs before intentional use.

The known Aircall manifest CRLF/LF hashing failure may still be reported. Fix it separately; do not disable validation. Test failures and timeouts are surfaced rather than treated as success.


In [ ]:
APPROVE_ACTION = ''
ACTIONS = {
    'targeted-tests': ['-m', 'pytest', 'tests/test_lead_capture.py', 'tests/test_conversation_service.py', 'tests/test_web_agent_p2.py', '-q'],
    'research-tests': ['-m', 'pytest', 'tests/test_product_research.py', '-q'],
    'full-tests': ['-m', 'pytest', '-q'],
    'evaluate-conversations': ['scripts/eval_customer_conversations.py'],
    'knowledge-overview': ['scripts/knowledge_workflow.py', 'overview'],
    'knowledge-validation': ['scripts/knowledge_workflow.py', 'validation'],
    'function-map': ['scripts/knowledge_function_map.py'],
    'source-audit-preview': ['scripts/review_local_sources.py'],
    'validate-catalogue': ['scripts/validate_catalogue.py'],
    'validate-aircall': ['scripts/validate_aircall_pack.py'],
    'validate-generated': ['scripts/check_generated_artifacts.py'],
    'retrieval-preview': ['scripts/build_retrieval_cards.py'],
    'legacy-rebuild-catalogue': ['scripts/build_family_sqlite.py'],
    'missing-source-preview': ['scripts/build_missing_tds_report.py'],
}

def local_environment():
    env = os.environ.copy()
    env.update(AGENT_USE_LLM='false', USE_HYBRID_RANKING='false', OLLAMA_HOST='http://127.0.0.1:11434', PYTHONIOENCODING='utf-8')
    return env

def run_action(action):
    if action not in ACTIONS:
        raise ValueError(f'Unknown action: {action}')
    if APPROVE_ACTION != action:
        print(f'Disabled. Set APPROVE_ACTION to {action!r} to run this action.')
        return
    command = [sys.executable, *[arg.replace('/', os.sep) for arg in ACTIONS[action]]]
    result = subprocess.run(command, cwd=ROOT, env=local_environment(), capture_output=True, text=True, encoding='utf-8', timeout=600, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()


In [ ]:
# Change this selector to one of the fixed actions above.
run_action('targeted-tests')


## Optional local chat server

Set `APPROVE_CHAT = 'start-local-chat'` to launch the chat on loopback port 8001, with model calls disabled. If that port is occupied, leave the existing process alone; this notebook never terminates someone else's server. The kernel owns the launched process: stop it with the next cell before closing the kernel. For a persistent manually run server, use the PowerShell command in `DEMO_CHAT_CHEATSHEET.md` instead.

Synthetic customer data only: lead storage still lacks production encryption/retention controls. Server logs go to the notebook output. No private admin key or lead details are displayed.


In [ ]:
APPROVE_CHAT = ''
if 'CHAT_PROCESS' not in globals():
    CHAT_PROCESS = None

if APPROVE_CHAT != 'start-local-chat':
    print('Chat startup disabled.')
elif CHAT_PROCESS is not None and CHAT_PROCESS.poll() is None:
    print('This notebook already owns a running chat process:', CHAT_PROCESS.pid)
else:
    import socket
    import time
    import urllib.request
    with socket.socket() as probe:
        probe.bind(('127.0.0.1', 8001))
    env = local_environment()
    env.update(AURORA_ENV='development', AURORA_ENABLE_DEMO_CHAT='true', AURORA_RATE_LIMIT_BACKEND='sqlite', AURORA_SESSION_BACKEND='sqlite')
    CHAT_PROCESS = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'web_agent:app', '--host', '127.0.0.1', '--port', '8001'], cwd=ROOT, env=env)
    try:
        for attempt in range(30):
            if CHAT_PROCESS.poll() is not None:
                raise RuntimeError(f'Chat exited with code {CHAT_PROCESS.returncode}; inspect server logs.')
            try:
                with urllib.request.urlopen('http://127.0.0.1:8001/chat', timeout=1) as response:
                    if response.status == 200:
                        print('Ready: http://127.0.0.1:8001/chat | PID:', CHAT_PROCESS.pid)
                        break
            except (urllib.error.URLError, TimeoutError):
                time.sleep(1)
        else:
            raise TimeoutError('Chat did not become ready; inspect server logs.')
    except Exception:
        if CHAT_PROCESS.poll() is None:
            CHAT_PROCESS.terminate()
            CHAT_PROCESS.wait(timeout=10)
        raise


In [ ]:
APPROVE_STOP = ''
if APPROVE_STOP == 'stop-notebook-chat' and CHAT_PROCESS is not None and CHAT_PROCESS.poll() is None:
    CHAT_PROCESS.terminate()
    CHAT_PROCESS.wait(timeout=10)
    print('Stopped the chat process started by this notebook.')
else:
    print('No stop requested; other servers are never stopped by this notebook.')


## Handover and next priorities

- Product Research is family-first at `/admin/products`, with child SKUs, source pages, existing manual inputs, immutable named-reviewer notes and separate preview/publish. Create accounts interactively using `scripts\research_accounts.py`; no default passwords. Approval alone does not change chatbot facts.
- Preserve all downloaded source files and provenance; remaining family links are pending owner input. Refresh reads existing results only; do not regenerate edited manual workbooks.
- Address cross-platform Aircall manifest hashing without weakening its validator.
- Investigate conversation routing and the two flagged evidence identities before changing recommendations.
- Add approved encryption, retention, consent and operator-access controls before real customer leads.
- Keep ordinary engineering deterministic; use already-installed local models selectively and serially. Historical Gemini tools must not be run.

Project documents (relative to this notebook): [Handover](../LOCAL_AGENT_HANDOVER.md), [Chat cheatsheet](../DEMO_CHAT_CHEATSHEET.md), [Policy](../BOT_POLICY.md), [Data security](../AUDIT_SECURITY.md), [README](../README.md).
